# 🚀 Google Colab GPU Server for Antigravity & Ollama
This notebook turns your Google Colab instance (Free T4 GPU or Pro A100/L4) into a remote high-speed AI backend for **Antigravity**.

### Features:
- Runs **Ollama** with full NVIDIA CUDA acceleration.
- Exposes the API endpoint securely to your local machine using a free **Cloudflare Tunnel** (`trycloudflare.com`).
- OpenAI-compatible API at `https://<tunnel-id>.trycloudflare.com/v1`.

### Step 1: Verify GPU Acceleration
Ensure you have set **Runtime > Change runtime type > Hardware accelerator > T4 GPU**.

In [ ]:
!nvidia-smi

### Step 2: Install Ollama & Cloudflared

In [ ]:
# Install zstd prerequisite & Ollama
!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh

# Download and install Cloudflare Tunnel (cloudflared)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1


### Step 3: Start Ollama Daemon in the Background

In [ ]:
import subprocess
import time

# Start Ollama in background listening on all interfaces
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    env=dict(subprocess.os.environ, OLLAMA_HOST="0.0.0.0:11434", OLLAMA_ORIGINS="*")
)
time.sleep(3)
print("✅ Ollama server is running on port 11434!")

### Step 4: Pull Your Desired Models
Choose which models to load into GPU memory:

In [ ]:
# Pull models (choose what you need, e.g. qwen3-vl:4b, qwen2.5:7b, deepseek-r1:8b, llama3.2:3b)
!ollama pull qwen3-vl:4b
!ollama list

### Step 5: Start Cloudflare Tunnel & Get Public URL
Run this cell. It will print your public `https://....trycloudflare.com` link to paste into Antigravity.

In [ ]:
import re
import subprocess
import urllib.request
import json

# Start cloudflared tunnel to localhost:11434
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:11434"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("Connecting Cloudflare Tunnel...")
colab_url = None
for line in tunnel_proc.stdout:
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        colab_url = match.group(0)
        print("\n" + "="*60)
        print("🎉 YOUR GOOGLE COLAB OLLAMA URL IS:")
        print(colab_url)
        print("="*60)
        print(f"\nOpenAI Base URL: {colab_url}/v1")
        print(f"API Tags:        {colab_url}/api/tags")
        
        # Auto-register with n8n
        try:
            req = urllib.request.Request(
                "https://15.252.236.234.sslip.io/webhook/colab-register",
                data=json.dumps({"ollama_tunnel_url": colab_url, "model": "qwen3-vl:4b"}).encode("utf-8"),
                headers={"Content-Type": "application/json"}
            )
            urllib.request.urlopen(req, timeout=5)
            print("⚡ Successfully registered with n8n Master Webhook!")
        except Exception:
            pass
        break


### Step 6: Keep-Alive Loop
Keep this cell running to maintain the server connection.

In [ ]:
import time
print("🟢 Colab GPU server active. Leave this running while working with Antigravity.")
try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print("Stopping server...")
    tunnel_proc.terminate()
    ollama_process.terminate()